# D2VLM on Kaggle — environment, model & asset setup

This notebook follows the official D2VLM README (https://github.com/nusnlp/d2vlm).
It only performs the steps that are impractical to prepare by hand:

1. Build the strict **Python 3.12 conda environment** (torch 2.4.0 / transformers 4.44.2 / nncore 0.4.5 / decord) as requested by the README.
2. Download the **D2VLM model weights** (`wenzhengzeng/D2VLM-Models`, ~10 GB) — too large to upload.
3. Download the **annotation-only** assets that are needed but too fiddly to upload:
   - `wenzhengzeng/D2VLM-Dataset` (`ETBench/evi` inference annotations)
   - `PolyU-ChenLab/ETBench` annotations + `evaluation/subset.json` (NOT the videos)
   - `ShuhuaiRen/TimeIT` Charades-STA / Youcook2 caption annotations
4. Pre-cache `sentence-transformers/all-MiniLM-L6-v2` (used by the metrics).
5. Verify the environment and provide ready-to-run inference/eval cells.

### Kaggle prerequisites (set in the notebook sidebar)
- **Accelerator:** GPU **T4 x2**
- **Internet:** **On** (needed for the downloads below)
- Attach your uploaded Kaggle datasets for the raw/compressed videos:
  - E.T. Bench compressed videos (`videos_compressed/`)
  - Charades-STA + Youcook2 post-processed root (`charades/`, `youcook2/`)

> The E.T. Bench video kit is ~149 GB, so it must be attached as a Kaggle Dataset or pre-compressed; the notebook downloads annotations only.

> The config cell below sets environment variables that the later `%%bash` cells inherit, so run the cells **in order**.

In [ ]:
%%bash
# Update package lists and install Python 3.10
apt-get update -y
apt-get install -y python3.12 python3.12-venv python3.12-dev

# Download and install pip for Python 3.10
wget -q https://bootstrap.pypa.io/get-pip.py
python3.12 get-pip.py --quiet
rm get-pip.py

In [ ]:
%%bash
# Create virtual environment using python3.12
python3.12 -m venv /kaggle/working/py312env

# Upgrade pip inside the python3.12 environment
/kaggle/working/py312env/bin/pip install --upgrade pip



In [ ]:
!nvidia-smi
!df -h /kaggle/working /kaggle/temp
!python --version


## 1. Build the Python 3.12 conda environment

In [ ]:
%%bash
# set -e
# source /opt/conda/etc/profile.d/conda.sh
# conda activate d2vlm
# PyTorch 2.4.0 (CUDA 12.1 wheels, matching the README)
/kaggle/working/py312env/bin/pip install --no-cache-dir torch==2.4.0 torchvision==0.19.0 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
%%bash
set -e
# source /opt/conda/etc/profile.d/conda.sh
# conda activate d2vlm

# Pinned stack from requirements.txt (deepspeed omitted: training-only, not needed for inference)
/kaggle/working/py312env/bin/pip install --no-cache-dir \
  numpy==1.26.4 \
  transformers==4.44.2 \
  accelerate==0.33.0 \
  peft==0.12.0 \
  sentence-transformers==3.0.1 \
  sentencepiece==0.2.0 \
  nncore==0.4.5 \
  tabulate psutil tqdm debugpy huggingface_hub hf_transfer

/kaggle/working/py312env/bin/pip install --no-cache-dir 'git+https://github.com/salaniz/pycocoevalcap.git@a24f74c'

# decord==0.6.0 has no cp312 wheel; fall back to eva-decord (drop-in decord API)
/kaggle/working/py312env/bin/pip install --no-cache-dir decord==0.6.0 || pip install --no-cache-dir eva-decord

In [ ]:
%%bash
# apt-get update -y
apt-get install -y openjdk-17-jdk-headless
java -version

In [ ]:
import os

# Set Java home directory
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["PATH"] = f"{os.environ['JAVA_HOME']}/bin:{os.environ['PATH']}"

# Verify Python can detect Java 17
import subprocess
print(subprocess.check_output(["java", "-version"], stderr=subprocess.STDOUT).decode())


## 2. Configure paths

Edit the `/kaggle/input/...` paths to match the Kaggle datasets you attach. The variables set here are inherited by every later `%%bash` cell.

In [ ]:
import os

# ---- working directories (persisted under /kaggle/working) ----
os.environ['KAGGLE_WORK'] = '/kaggle/working'
os.environ['D2VLM_ROOT'] = '/kaggle/working/d2vlm'          # repo clone
os.environ['MODELS_DIR'] = '/kaggle/working/models'         # downloaded HF weights
os.environ['ANNOS_DIR']  = '/kaggle/working/annotations'    # downloaded HF annotations
os.environ['HF_HOME']    = '/kaggle/working/hf_cache'       # persisted HF cache
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

# Put the d2vlm conda env first on PATH so !cmd / %%bash use Python 3.12
env_bin = '/opt/conda/envs/d2vlm/bin'
os.environ['PATH'] = env_bin + os.pathsep + os.environ.get('PATH', '')
os.environ['CONDA_PREFIX'] = '/opt/conda/envs/d2vlm'

# ==== ATTACH THESE AS KAGGLE DATASETS (edit the /kaggle/input/... paths) ====
os.environ['ETBENCH_VIDEOS'] = '/kaggle/input/etbench-videos-compressed/videos_compressed'
os.environ['OTHER_BENCH_ROOT'] = '/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize'  # contains charades/ and youcook2/
os.environ['PRED_ROOT'] = '/kaggle/working/preds'


# ---- model selection: SFT `d2vlm` or `d2vlm_mcqa_enhanced` ----
os.environ['MODEL_SUBDIR'] = 'd2vlm'        # 'd2vlm_mcqa_enhanced' for the MCQA-enhanced model
os.environ['ARCH_MODULE'] = 'd2vlm_arch'    # 'd2vlm_fpo_arch' for the MCQA-enhanced model

# ---- annotation paths (downloaded below; set here so later cells can use them) ----
os.environ['D2VLM_DATASET'] = os.path.join(os.environ['ANNOS_DIR'], 'D2VLM-Dataset')
os.environ['ETBENCH_OFFICIAL'] = os.path.join(os.environ['ANNOS_DIR'], 'ETBench-official')
os.environ['TIMEIT_DIR'] = os.path.join(os.environ['ANNOS_DIR'], 'TimeIT')
os.environ['CHARADES_GT'] = os.path.join(os.environ['TIMEIT_DIR'], 'data', 'temporal_video_grounding', 'charades', 'charades_annotation', 'test.caption_coco_format.json')
os.environ['YOUCOOK2_GT'] = os.path.join(os.environ['TIMEIT_DIR'], 'data', 'dense_video_captioning', 'youcook2', 'val.caption_coco_format.json')

for d in [os.environ['MODELS_DIR'], os.environ['ANNOS_DIR'], os.environ['HF_HOME'], os.environ['PRED_ROOT']]:
    os.makedirs(d, exist_ok=True)

for k in ['D2VLM_ROOT', 'MODELS_DIR', 'ANNOS_DIR', 'HF_HOME', 'ETBENCH_VIDEOS',
          'OTHER_BENCH_ROOT', 'PRED_ROOT', 'MODEL_SUBDIR', 'ARCH_MODULE']:
    print(k, '=', os.environ[k])

## 3. Clone the D2VLM repository

In [ ]:
%%bash
set -e
if [ ! -d $D2VLM_ROOT/.git ]; then
  git clone https://github.com/nusnlp/d2vlm.git $D2VLM_ROOT
else
  echo 'repo already present'
fi
cd $D2VLM_ROOT && git log --oneline -1

## 4. Download the model weights (cannot be uploaded by hand)

`wenzhengzeng/D2VLM-Models` -> only the `d2vlm*/` folder selected in the config cell. The repo also ships TF events/runs/etbench folders that are not needed for inference and are skipped here.

In [ ]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
import os
from huggingface_hub import snapshot_download

sub = os.environ['MODEL_SUBDIR']
snapshot_download(
    repo_id='wenzhengzeng/D2VLM-Models',
    local_dir=os.environ['MODELS_DIR'],
    allow_patterns=[
        f'{sub}/config.json',
        f'{sub}/generation_config.json',
        f'{sub}/model-*.safetensors',
        f'{sub}/model.safetensors.index.json',
        f'{sub}/tokenizer.json',
        f'{sub}/tokenizer.model',
        f'{sub}/tokenizer_config.json',
        f'{sub}/special_tokens_map.json',
        f'{sub}/added_tokens.json',
    ],
    max_workers=8,
)
print('model ready at', os.path.join(os.environ['MODELS_DIR'], sub))
PY

## 5. Download the annotation assets (annotations are small; videos are attached separately)

In [ ]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
import os
from huggingface_hub import snapshot_download

# D2VLM inference annotations for E.T. Bench (ETBench/evi)
snapshot_download(
    repo_id='wenzhengzeng/D2VLM-Dataset',
    repo_type='dataset',
    local_dir=os.environ['D2VLM_DATASET'],
    allow_patterns=['ETBench/*'],
)
print('D2VLM-Dataset annotations ready')

# Official E.T. Bench annotations + subset.json (needed by --subset) -- NOT the videos
snapshot_download(
    repo_id='PolyU-ChenLab/ETBench',
    repo_type='dataset',
    local_dir=os.environ['ETBENCH_OFFICIAL'],
    allow_patterns=[
        'annotations/*',
        'evaluation/subset.json',
        'evaluation/compute_metrics.py',
        'compress_videos.py',
        'README.md',
    ],
)
print('ETBench annotations ready')

# TimeIT annotations for Charades-STA and Youcook2
snapshot_download(
    repo_id='ShuhuaiRen/TimeIT',
    repo_type='dataset',
    local_dir=os.environ['TIMEIT_DIR'],
    allow_patterns=[
        'data/temporal_video_grounding/charades/charades_annotation/test.caption_coco_format.json',
        'data/dense_video_captioning/youcook2/val.caption_coco_format.json',
    ],
)
print('TimeIT annotations ready')
PY

## 6. Pre-cache the sentence-transformers model used by the metrics

`all_benchmark_eval/etbench/new_compute_metrics.py` loads `sentence-transformers/all-MiniLM-L6-v2`. Caching it now avoids a network fetch during evaluation (and works around the hard-coded `hf-mirror.com` endpoint).

In [ ]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
from sentence_transformers import SentenceTransformer
SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
print('cached all-MiniLM-L6-v2')
PY

## 7. Verify the environment

In [ ]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 - <<'PY'
import importlib.metadata as md
import torch
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| gpus', torch.cuda.device_count())
import transformers
print('transformers', transformers.__version__)
for p in ['decord', 'eva-decord', 'nncore', 'peft', 'sentence-transformers', 'pycocoevalcap']:
    try:
        print(p, md.version(p))
    except Exception:
        pass
import decord, nncore  # noqa: F401
print('imports OK')
PY

In [ ]:
%%bash
set -e
/kaggle/working/py312env/bin/python3.12 --version

## 8. Run inference & evaluation

These cells call the exact entry points from the README. They are run as `python` (env 3.12) because some scripts use Python 3.12-only f-strings. `scripts/inference.sh` is reproduced here with Kaggle paths and 2 T4 chunks.

> The Youcook2/MVBench/Video-MME `.sh` wrappers in the repo contain a typo (`etchat/eval/...` instead of `d2vlm/eval/...`) and hard-coded `/storage/...` paths, so the cells below invoke the corrected `d2vlm/eval/infer_etbench_decouple.py` directly. No repository file is modified.

### 8a. Smoke test (20 E.T. Bench samples)

In [ ]:
# %%bash
# set -e
# python - <<'PY'
# import os, glob, nncore
# anno_dir = os.path.join(os.environ['D2VLM_DATASET'], 'ETBench', 'evi')
# files = sorted(glob.glob(os.path.join(anno_dir, '*.json')))
# print('evi files:', [os.path.basename(f) for f in files])
# samples = []
# for f in files:
#     samples += nncore.load(f)
# print('total evi samples:', len(samples))
# os.makedirs('/kaggle/working/smoke_anno', exist_ok=True)
# nncore.dump(samples[:20], '/kaggle/working/smoke_anno/smoke.json')
# print('wrote 20 smoke samples')
# PY

In [ ]:
# %%bash
# set -e
# cd $D2VLM_ROOT
# export PYTHONPATH="./:$PYTHONPATH"
# SMOKE_DIR="$PRED_ROOT/smoke/etbench"
# mkdir -p "$SMOKE_DIR"
# CUDA_VISIBLE_DEVICES=0 MASTER_PORT=29510 python d2vlm/eval/infer_etbench_decouple.py \
#   --anno_path /kaggle/working/smoke_anno/smoke.json \
#   --data_path "$ETBENCH_VIDEOS" \
#   --pred_path "$SMOKE_DIR" \
#   --model_path "$MODELS_DIR/$MODEL_SUBDIR" \
#   --arch_module "$ARCH_MODULE" \
#   --chunk 1 --index 0 --verbose
# python all_benchmark_eval/etbench/new_compute_metrics.py --pred_path "$SMOKE_DIR"

### 8b. E.T. Bench — full run (2 chunks, one per T4)

For the full 7,289-sample set this is long; the chunk outputs (`etbench_0.json`, `etbench_1.json`) are written to `$PRED_ROOT/etbench`. If a session ends early, commit `/kaggle/working` and re-run only the missing chunk before computing metrics.

In [ ]:
# %%bash
# set -e
# cd $D2VLM_ROOT
# export PYTHONPATH="./:$PYTHONPATH"
# PRED_DIR="$PRED_ROOT/etbench"
# mkdir -p "$PRED_DIR"
# for IDX in 0 1; do
#   CUDA_VISIBLE_DEVICES=$IDX MASTER_PORT=$((29502+IDX)) python d2vlm/eval/infer_etbench_decouple.py \
#     --anno_path "$D2VLM_DATASET/ETBench/evi" \
#     --data_path "$ETBENCH_VIDEOS" \
#     --pred_path "$PRED_DIR" \
#     --model_path "$MODELS_DIR/$MODEL_SUBDIR" \
#     --arch_module "$ARCH_MODULE" \
#     --chunk 2 --index $IDX &
# done
# wait
# python all_benchmark_eval/etbench/new_compute_metrics.py --pred_path "$PRED_DIR"

### 8c. Charades-STA

Requires the post-processed root (`charades/compressed_videos`, `charades/inference_input.json`) produced by `other_benchmark_organize/charades/run.sh`, attached at `$OTHER_BENCH_ROOT`.

In [ ]:
%%bash
set -e
cd $D2VLM_ROOT
export PYTHONPATH="./:$PYTHONPATH"
PRED_DIR="$PRED_ROOT/charades"
mkdir -p "$PRED_DIR"
for IDX in 0 1; do
  CUDA_VISIBLE_DEVICES=$IDX MASTER_PORT=$((29502+IDX)) /kaggle/working/py312env/bin/python3.12 d2vlm/eval/infer_etbench_decouple.py \
    --anno_path "/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/charades/inference_input.json" \
    --data_path "$OTHER_BENCH_ROOT" \
    --pred_path "$PRED_DIR" \
    --model_path "$MODELS_DIR/$MODEL_SUBDIR" \
    --arch_module "$ARCH_MODULE" \
    --chunk 2 --index $IDX &
done
wait
/kaggle/working/py312env/bin/python3.12 all_benchmark_eval/charades/convert_result_in_timechat_format.py --pred_path "$PRED_DIR"
/kaggle/working/py312env/bin/python3.12 all_benchmark_eval/charades/eval_in_timechat_format.py \
  --pred_file "$PRED_DIR/result_converted.json" \
  --gt_file "$CHARADES_GT"



/kaggle/input/datasets/nguynhongthunpht/d2vlm-charades-youcook2/other_benchmark_organize/charades/inference_input.json

/kaggle/input/datasets/nguynhongthunpht/d2vlm_charades_youcook2/other_benchmark_organize/charades/inference_input.json

/kaggle/input/datasets/nguynhongthunpht/d2vlm_charades_youcook2/other_benchmark_organize/charades/inference_input.json
/kaggle/input/datasets/nguynhongthunpht/d2vlm-charades-youcook2/other_benchmark_organize/charades/inference_input.json
/kaggle/input/datasets/nguynhongthunpht/d2vlm-charades-youcook2/other_benchmark_organize/charades/inference_input.json

/kaggle/input/datasets/nguynhongthunpht/d2vlm-charades-youcook2/other_benchmark_organize/charades/compressed_videos/0FJSU.mp4
/kaggle/input/datasets/nguynhongthunpht/d2vlm_charades_youcook2/other_benchmark_organize/charades/compressed_videos/3MSZA.mp4,
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/inference_input.json
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/val.caption_coco_format.json

### 8d. Youcook2

Requires the post-processed root (`youcook2/compressed_videos`, `youcook2/inference_input.json`) produced by `other_benchmark_organize/youcook2/run.sh`, attached at `$OTHER_BENCH_ROOT`. Needs Java (installed above) for METEOR.

Can't be run currently as the videos as missing (downloaded from llms-eval/youcook2), the original downloading script has missing videos

In [ ]:
%%bash
set -e
cd $D2VLM_ROOT
export PYTHONPATH="./:$PYTHONPATH"
PRED_DIR="$PRED_ROOT/youcook2"
mkdir -p "$PRED_DIR"
for IDX in 0 1; do
  CUDA_VISIBLE_DEVICES=$IDX MASTER_PORT=$((29502+IDX)) /kaggle/working/py312env/bin/python3.12 d2vlm/eval/infer_etbench_decouple.py \
    --anno_path "/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/inference_input.json" \
    --data_path "$OTHER_BENCH_ROOT" \
    --pred_path "$PRED_DIR" \
    --model_path "$MODELS_DIR/$MODEL_SUBDIR" \
    --arch_module "$ARCH_MODULE" \
    --chunk 2 --index $IDX &
done
wait
/kaggle/working/py312env/bin/python3.12 all_benchmark_eval/youcook2/convert_result_in_timechat_format.py --pred_path "$PRED_DIR"
/kaggle/working/py312env/bin/python3.12 all_benchmark_eval/youcook2/eval_in_timechat_format.py \
  --pred_file "$PRED_DIR/result_converted.json" \
  --gt_file "$YOUCOOK2_GT"

In [ ]:
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/compressed_videos/V53XmPeyjIU.mp4\
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/compressed_video/V53XmPeyjIU.mp4

/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/compressed_videos/NZtwPf32YN4.mp4
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/compress_convert_mp4_cp.py
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/inference_input.json
/kaggle/input/datasets/nguynhongthunpht/other-benchmark-organize/other_benchmark_organize/youcook2/inference_input.json



## Notes / known issues

- **Videos are not downloaded here.** Attach them as Kaggle Datasets and point `ETBENCH_VIDEOS` / `OTHER_BENCH_ROOT` at them. The E.T. Bench kit is ~149 GB, so it must be pre-compressed or attached as a dataset.
- **decord**: `decord==0.6.0` has no Python 3.12 wheel; the install falls back to `eva-decord` (same `decord` module API).
- **nncore 0.4.5** compiles a small CUDA op (`temporal_iou`) against the installed torch during `pip install`; that is expected and may take a few minutes.
- **deepspeed** is intentionally not installed (training-only; not used for inference).
- **METEOR** needs Java; installed via conda-forge `openjdk=17`.
- The metrics scripts hard-code `HF_ENDPOINT=https://hf-mirror.com`; the sentence-transformers model is pre-cached above, so evaluation should load it locally. Set `MODEL_SUBDIR=d2vlm_mcqa_enhanced` and `ARCH_MODULE=d2vlm_fpo_arch` to use the MCQA-enhanced weights.
- **Persistence**: commit the notebook / save `Version` to keep `/kaggle/working` (model, preds, cache) between sessions. Model + outputs may approach Kaggle's ~20 GB working-disk limit; use `/kaggle/temp` for the model if space runs out (it is not persisted).
- E.T. Bench `--subset` additionally requires `all_benchmark_eval/etbench/subset.json`, available at `$ETBENCH_OFFICIAL/evaluation/subset.json`.